# yfinance로 주식 불러오기

In [247]:
import yfinance as yf

In [248]:
code = '010130.KS'

In [249]:
import datetime

In [250]:
start = datetime.datetime(2024, 12, 1)

In [251]:
start

datetime.datetime(2024, 12, 1, 0, 0)

In [252]:
end = datetime.datetime(2025, 3, 31)

In [253]:
end

datetime.datetime(2025, 3, 31, 0, 0)

In [254]:
df = yf.download(code, start, end)

/tmp/ipykernel_6512/328843078.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(code, start, end)
[*********************100%***********************]  1 of 1 completed


In [255]:
df

Price,Close,High,Low,Open,Volume
Ticker,010130.KS,010130.KS,010130.KS,010130.KS,010130.KS
Date,,,,,
2024-12-02,1.380544e+06,1.500889e+06,1.201494e+06,1.208343e+06,133739
2024-12-03,1.508716e+06,1.606558e+06,1.389349e+06,1.418702e+06,116463
2024-12-04,1.634931e+06,1.682874e+06,1.545896e+06,1.569378e+06,85292
2024-12-05,1.956830e+06,1.956830e+06,1.674068e+06,1.693636e+06,95215
2024-12-06,1.773866e+06,2.355045e+06,1.698528e+06,2.078153e+06,250745
...,...,...,...,...,...
2025-03-24,8.062826e+05,8.348323e+05,8.062826e+05,8.269565e+05,13716
2025-03-25,8.112049e+05,8.171118e+05,8.052981e+05,8.082515e+05,14056


In [256]:
df = df['Close']['010130.KS']

In [257]:
df

,010130.KS
Date,
2024-12-02,1.380544e+06
2024-12-03,1.508716e+06
2024-12-04,1.634931e+06
2024-12-05,1.956830e+06
2024-12-06,1.773866e+06
...,...
2025-03-24,8.062826e+05
2025-03-25,8.112049e+05
2025-03-26,8.358168e+05


# 네이버 주식 크롤링
- 기사제목 --> 어떤 tag에 쓰여 있는지 알아내기 --> 브라우저의 "개발자 도구 (F12)"를 켜서 찾기
- 기사 본문 링크
- 언론사 이름
- 다음 페이지 수

In [258]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [259]:
base_url = 'https://finance.naver.com/news/news_list.naver'

In [260]:
params_base = {'mode':'LSS2D', 'section_id':'101', 'section_id2':'258'}

In [261]:
page=1
params={**params_base, 'page':page}

In [262]:
params

{'mode': 'LSS2D', 'section_id': '101', 'section_id2': '258', 'page': 1}

In [263]:
response = requests.get(base_url, params=params)

In [264]:
response # 200 정상적으로 끝마쳤다 HTML 상태

<Response [200]>

In [265]:
response.text

'<!--  global include -->\n\n\t\n\t\n\t\n\t\n<html lang=\'ko\'>\n<head>\n\n\n\t\n\t\n\t\t\n\t\t\t\n\t\t\t\t<title>Npay 증권</title>\n\t\t\t\n\t\t\t\n\t\t\n\t\n\n\n\n\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n\n<meta http-equiv="Content-Script-Type" content="text/javascript">\n<meta http-equiv="Content-Style-Type" content="text/css">\n<meta name="apple-mobile-web-app-title" content="Npay 증권" />\n\n\n\n\n\n\t\n    \n        <meta property="og:url" content="http://finance.naver.com/news/news_list.naver?mode=LSS2D&section_id=101&section_id2=258&type=0"/>\n        \n\t\t\t\n\t\t    \n\t\t    \t<meta property="og:title" content="실시간 속보 : Npay 증권"/>\n\t\t     \n\t\t\n\t\t\n\t\t\t\n\t\t\t   <meta property="og:description" content="관심종목의 실시간 주가를 가장 빠르게 확인하는 곳"/>\n\t\t    \n\t\t    \n\t\t\n\t\t \n\t\t\t\n\t\t\t    <meta property="og:image" content="https://ssl.pstatic.net/static/m/stock/im/2016/08/og_stock-200.png"/>\n\t\t    \n\t\t    \n\t\t\n    \n\n<meta property="

In [266]:
soup = bs(response.text, 'html.parser') # html 문서로 고쳐준다

## bs
: 주어진 텍스트 구조를 분석해서
- html & xml 문서처럼 사용할 수 있게 해준다.
- 크롤링 기능 제공 => 이걸 보고 parsing이라 함

In [267]:
soup.select('.articleSubject')

[<dd class="articleSubject">
 <a href="/news/news_read.naver?article_id=0006257135&amp;office_id=018&amp;mode=LSS2D&amp;type=0§ion_id=101§ion_id2=258§ion_id3=&amp;date=20260414&amp;page=1" title="[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감">[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감</a>
 </dd>,
 <dd class="articleSubject">
 <a href="/news/news_read.naver?article_id=0004610586&amp;office_id=011&amp;mode=LSS2D&amp;type=0§ion_id=101§ion_id2=258§ion_id3=&amp;date=20260414&amp;page=1" title="위험선호 확산 vs 결제수요…환율 1480원대 공방 [김혜란의 FX]">위험선호 확산 vs 결제수요…환율 1480원대 공방 [김혜란의 FX]</a>
 </dd>,
 <dd class="articleSubject">
 <a href="/news/news_read.naver?article_id=0008889145&amp;office_id=421&amp;mode=LSS2D&amp;type=0§ion_id=101§ion_id2=258§ion_id3=&amp;date=20260414&amp;page=1" title="코스피,  2.74% 상승한 5967.75 마감…코스닥 2.00%↑(2보)">코스피,  2.74% 상승한 5967.75 마감…코스닥 2.00%↑(2보)</a>
 </dd>,
 <dd class="articleSubject">
 <a href="/news/news_read.naver?article_id=0005274805&amp;office_id=015&amp;mode=LSS2D&amp;typ

In [268]:
type(soup.select('.articleSubject'))

bs4.element.ResultSet

In [269]:
articlesubject_list = soup.select('.articleSubject')
len(articlesubject_list)

20

In [270]:
articlesubject_list[0]

<dd class="articleSubject">
<a href="/news/news_read.naver?article_id=0006257135&amp;office_id=018&amp;mode=LSS2D&amp;type=0§ion_id=101§ion_id2=258§ion_id3=&amp;date=20260414&amp;page=1" title="[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감">[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감</a>
</dd>

In [271]:
articlesubject_list[0].select_one('a').get_text()

'[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감'

In [272]:
for i in range(len(articlesubject_list)):
    title = articlesubject_list[i].select_one('a').get_text()
    print(title)

[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감
위험선호 확산 vs 결제수요…환율 1480원대 공방 [김혜란의 FX]
코스피,  2.74% 상승한 5967.75 마감…코스닥 2.00%↑(2보)
외국인 현·선물 2.7조 순매수에…'육천피' 턱밑까지
코스피, 2.74% 오른 5,967.75 마감…코스닥도 2%↑
[달러·원] 환율 8.1원 내 1481.2원 마감
[코스닥] 22.04p(2.00%) 오른 1121.88 마감
[코스피] 159.12p(2.74%) 오른 5967.74 마감
[속보] 6,000선 턱밑…SK하이닉스6%·스퀘어 10%↑
차바이오F&C, 반려동물 영양제 '펫세븐 장 케어 유산균' 출시
코스피, 장중 '6천피' 찍고 5,960선 마감…코스닥도 올라
코스피 5967.75(▲2.74%), 코스닥 1121.88(▲2.00%), 원·달러 환율 ..
[코스피] 159.13포인트(2.74%) 오른 5967.75 마감
[속보] 코스닥 1121.88…2.00% 상승 마감
키움證 로보어드바이저 '키움모멘텀', 1년 수익률 최고수준 기록
코스닥 22.04P 오른 1121.88 마감(2.0%↑)
[코스닥] 22.04포인트(2.00%) 오른 1121.88 마감
[속보]코스피, 협상 기대 속 2.74% 오른 5967.75 마감
코스피 159.13P 오른 5967.75 마감(2.74%↑)
[속보] 코스피 5967.74…2.74% 상승 마감


In [273]:
for i in articlesubject_list:
    title = i.select_one('a').get_text()
    print(title)

[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감
위험선호 확산 vs 결제수요…환율 1480원대 공방 [김혜란의 FX]
코스피,  2.74% 상승한 5967.75 마감…코스닥 2.00%↑(2보)
외국인 현·선물 2.7조 순매수에…'육천피' 턱밑까지
코스피, 2.74% 오른 5,967.75 마감…코스닥도 2%↑
[달러·원] 환율 8.1원 내 1481.2원 마감
[코스닥] 22.04p(2.00%) 오른 1121.88 마감
[코스피] 159.12p(2.74%) 오른 5967.74 마감
[속보] 6,000선 턱밑…SK하이닉스6%·스퀘어 10%↑
차바이오F&C, 반려동물 영양제 '펫세븐 장 케어 유산균' 출시
코스피, 장중 '6천피' 찍고 5,960선 마감…코스닥도 올라
코스피 5967.75(▲2.74%), 코스닥 1121.88(▲2.00%), 원·달러 환율 ..
[코스피] 159.13포인트(2.74%) 오른 5967.75 마감
[속보] 코스닥 1121.88…2.00% 상승 마감
키움證 로보어드바이저 '키움모멘텀', 1년 수익률 최고수준 기록
코스닥 22.04P 오른 1121.88 마감(2.0%↑)
[코스닥] 22.04포인트(2.00%) 오른 1121.88 마감
[속보]코스피, 협상 기대 속 2.74% 오른 5967.75 마감
코스피 159.13P 오른 5967.75 마감(2.74%↑)
[속보] 코스피 5967.74…2.74% 상승 마감


In [274]:
i.select_one('a')

<a href="/news/news_read.naver?article_id=0001138955&amp;office_id=417&amp;mode=LSS2D&amp;type=0§ion_id=101§ion_id2=258§ion_id3=&amp;date=20260414&amp;page=1" title="[속보] 코스피 5967.74…2.74% 상승 마감">[속보] 코스피 5967.74…2.74% 상승 마감</a>

In [275]:
'https://finance.naver.com'+i.select_one('a').get('href', '')

'https://finance.naver.com/news/news_read.naver?article_id=0001138955&office_id=417&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1'

In [276]:
for i in articlesubject_list:
    a_tag = i.select_one('a')
    title = a_tag.get_text(strip=True)
    link = 'https://finance.naver.com'+a_tag.get('href', '')
    print(title)
    print(link)
    print()

[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감
https://finance.naver.com/news/news_read.naver?article_id=0006257135&office_id=018&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1

위험선호 확산 vs 결제수요…환율 1480원대 공방 [김혜란의 FX]
https://finance.naver.com/news/news_read.naver?article_id=0004610586&office_id=011&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1

코스피,  2.74% 상승한 5967.75 마감…코스닥 2.00%↑(2보)
https://finance.naver.com/news/news_read.naver?article_id=0008889145&office_id=421&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1

외국인 현·선물 2.7조 순매수에…'육천피' 턱밑까지
https://finance.naver.com/news/news_read.naver?article_id=0005274805&office_id=015&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1

코스피, 2.74% 오른 5,967.75 마감…코스닥도 2%↑
https://finance.naver.com/news/news_read.naver?article_id=0000854953&office_id=422&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1

[달러·원] 환율 8.1원 내 1481.2원 마감
https:/

In [277]:
soup

<!--  global include -->
<html lang="ko">
<head>
<title>Npay 증권</title>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="text/javascript" http-equiv="Content-Script-Type"/>
<meta content="text/css" http-equiv="Content-Style-Type"/>
<meta content="Npay 증권" name="apple-mobile-web-app-title">
<meta content="http://finance.naver.com/news/news_list.naver?mode=LSS2D§ion_id=101§ion_id2=258&amp;type=0" property="og:url">
<meta content="실시간 속보 : Npay 증권" property="og:title"/>
<meta content="관심종목의 실시간 주가를 가장 빠르게 확인하는 곳" property="og:description"/>
<meta content="https://ssl.pstatic.net/static/m/stock/im/2016/08/og_stock-200.png" property="og:image"/>
<meta content="article" property="og:type"/>
<meta content="" property="og:article:thumbnailUrl"/>
<meta content="Npay 증권" property="og:article:author"/>
<meta content="http://FINANCE.NAVER.COM" property="og:article:author:url"/>
<link href="https://ssl.pstatic.net/imgstock/static.pc/20260406095636/css/finance_head

In [278]:
articleSummary_list = soup.select('.articleSummary')

In [279]:
articleSummary_list[0]

<dd class="articleSummary">
								
									
										[이데일리 김경은 기자]
									
									
								
								<span class="press">이데일리</span>
<span class="bar">|</span>
<span class="wdate">2026-04-14 15:41</span>
</dd>

In [280]:
for i in articleSummary_list:
    press_tag = i.select_one('.press')
    press = press_tag.get_text(strip=True)
    print(press)

이데일리
서울경제
뉴스1
한국경제
연합뉴스TV
뉴스1
뉴스1
뉴스1
한국경제TV
파이낸셜뉴스
연합뉴스
뉴시스
서울경제
동행미디어 시대
연합뉴스
아시아경제
서울경제
뉴시스
아시아경제
동행미디어 시대


In [281]:
for i, item in enumerate(articlesubject_list):
    a_tag = item.select_one('a')
    title = a_tag.get_text(strip=True)
    link = 'https://finance.naver.com'+a_tag.get('href', '')
    press = articleSummary_list[i].select_one('.press').get_text(strip=True)
    dummy_dict = {'제목':title, 'URL':link, '언론사':press, '페이지':page}
    print(dummy_dict)

{'제목': '[속보]코스피, 2.74% 오른 5967.75...코스닥, 2.0% 상승 마감', 'URL': 'https://finance.naver.com/news/news_read.naver?article_id=0006257135&office_id=018&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1', '언론사': '이데일리', '페이지': 1}
{'제목': '위험선호 확산 vs 결제수요…환율 1480원대 공방 [김혜란의 FX]', 'URL': 'https://finance.naver.com/news/news_read.naver?article_id=0004610586&office_id=011&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1', '언론사': '서울경제', '페이지': 1}
{'제목': '코스피,  2.74% 상승한 5967.75 마감…코스닥 2.00%↑(2보)', 'URL': 'https://finance.naver.com/news/news_read.naver?article_id=0008889145&office_id=421&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1', '언론사': '뉴스1', '페이지': 1}
{'제목': "외국인 현·선물 2.7조 순매수에…'육천피' 턱밑까지", 'URL': 'https://finance.naver.com/news/news_read.naver?article_id=0005274805&office_id=015&mode=LSS2D&type=0§ion_id=101§ion_id2=258§ion_id3=&date=20260414&page=1', '언론사': '한국경제', '페이지': 1}
{'제목': '코스피, 2.74% 오른 5,967.75 마감…코스닥도 2%↑', 'URL'

In [289]:
def parse_page(page):
    base_url = 'https://finance.naver.com/news/news_list.naver'
    base_params = {'mode':'LSS2D', 'section_id':'101', 'section_id2':'258'}
    params={**base_params, 'page':page}
    response = requests.get(base_url, params=params)
    soup = bs(response.text, 'html.parser')

    a_sub_list = soup.select('.articleSubject')
    a_smm_list = soup.select('.articleSummary')
    articles = []

    for i, item in enumerate(a_sub_list):
        a_tag = item.select_one('a')
        title = a_tag.get_text(strip=True)
        href = 'https://finance.naver.com'+a_tag.get('herf', '')
        press = a_smm_list[i].select_one('.press').get_text(strip=True)
        dummy_dict = {'제목':title, 'URL':href, '언론사':press, '페이지':page}
        articles.append(dummy_dict)
    return articles

In [290]:
articles = parse_page(2)

In [291]:
articles

[{'제목': '코스피, 장중 6000선 터치하고 5900선 마감',
  'URL': 'https://finance.naver.com',
  '언론사': '뉴시스',
  '페이지': 2},
 {'제목': '코스피 상승, 원·달러 환율 하락 마감',
  'URL': 'https://finance.naver.com',
  '언론사': '뉴스1',
  '페이지': 2},
 {'제목': '코스피, 6000선 턱밑에서 장 마감',
  'URL': 'https://finance.naver.com',
  '언론사': '뉴스1',
  '페이지': 2},
 {'제목': "9일간 '147조' 터졌다…주가 급등에 개미들 환호한 종목",
  'URL': 'https://finance.naver.com',
  '언론사': '한국경제',
  '페이지': 2},
 {'제목': '코스피·코스닥 상승, 원·달러 환율 하락 마감',
  'URL': 'https://finance.naver.com',
  '언론사': '뉴스1',
  '페이지': 2},
 {'제목': "협상 기대감에 '육천피' 눈앞…외국인·기관 '쌍끌이' 매수",
  'URL': 'https://finance.naver.com',
  '언론사': '연합뉴스TV',
  '페이지': 2},
 {'제목': "중동 종전 기대감에 '증시 상승·환율 하락'",
  'URL': 'https://finance.naver.com',
  '언론사': '뉴스1',
  '페이지': 2},
 {'제목': "코스피, 장중 '6000피' 재돌파 후 5960선 마감",
  'URL': 'https://finance.naver.com',
  '언론사': '뉴스1',
  '페이지': 2},
 {'제목': '코스피·코스닥 동반 상승',
  'URL': 'https://finance.naver.com',
  '언론사': '뉴스1',
  '페이지': 2},
 {'제목': '상승 마감한 국내 증시',
  'URL': 'https://finance.naver.com',

In [296]:
import time

In [306]:
def finance_news(max_pages=3, delay=1.0):
    all_articles = []
    for page in range(1, max_pages+1):
        print(f"{page} / {max_pages}번째 페이지 수집 중...")
        try:
            articles = parse_page(page)
            all_articles.extend(articles)
        except Exception as e:
            print(e)
        if page < max_pages:
            time.sleep(delay)
    df = pd.DataFrame(all_articles)
    print(f"총 {len(df)} 건 수집 완료\n")
    return df

In [307]:
finance_news(3)

1 / 3번째 페이지 수집 중...
2 / 3번째 페이지 수집 중...
3 / 3번째 페이지 수집 중...
총 60 건 수집 완료



,제목,URL,언론사,페이지
0,"하나운용, ETF 보수 인상에도 “최저” 홍보…당국 기조 ‘역행’[only 이데일리]",https://finance.naver.com,이데일리,1
1,"NH증권, '예탁자산 300억' 패밀리오피스 세미나 개최",https://finance.naver.com,뉴시스,1
2,[뉴스1PICK]코스피 장중 6000선 터치 후 5967선 마감…반도체주 강세 영향,https://finance.naver.com,뉴스1,1
3,"[특징주] LS일렉트릭, 대규모 전력 인프라 수주에 52주 신고가(종합)",https://finance.naver.com,연합뉴스,1
4,"큐라티스, 9대1 주식병합 감자 결정",https://finance.naver.com,이데일리,1
5,[단독] ‘K-푸드’ 본촌치킨 매각 급물살…본입찰 착수,https://finance.naver.com,헤럴드경제,1
6,"""이익 1000조 시대""…코스피 7500선 '현실화 구간' 진입",https://finance.naver.com,파이낸셜뉴스,1
7,"'HBM 12단도 넘본다'…中반도체 공룡, 'K-초격차' 흔들까",https://finance.naver.com,비즈워치,1
8,AI 인프라 투자 기대 재점화…LS일렉트릭 신고가 터치[핫종목](종합),https://finance.naver.com,뉴스1,1
9,"에스바이오메딕스, 400억 규모 자금 조달",https://finance.naver.com,매일경제,1


In [303]:
finance_news(10)

총 200 건 수집 완료



,제목,URL,언론사,페이지
0,"가상자산거래, 업비트만 위험관리 ‘합격’",https://finance.naver.com,매일경제,1
1,"하나운용, ETF 보수 인상에도 “최저” 홍보…당국 기조 ‘역행’[only 이데일리]",https://finance.naver.com,이데일리,1
2,"NH증권, '예탁자산 300억' 패밀리오피스 세미나 개최",https://finance.naver.com,뉴시스,1
3,[뉴스1PICK]코스피 장중 6000선 터치 후 5967선 마감…반도체주 강세 영향,https://finance.naver.com,뉴스1,1
4,"[특징주] LS일렉트릭, 대규모 전력 인프라 수주에 52주 신고가(종합)",https://finance.naver.com,연합뉴스,1
...,...,...,...,...
195,"최영훈 채비 대표 ""전기차 급속충전 호황 타고 실적 개선 이뤄내겠다""",https://finance.naver.com,동행미디어 시대,10
196,'3高' 불청객이 왔다...진짜 관심주는 따로 있다는데 [마켓딥다이브],https://finance.naver.com,한국경제TV,10
197,스페이스X 상장 수혜...TIGER 미국우주테크 ETF 상장,https://finance.naver.com,머니투데이,10
198,"키움증권, LG·런던거래소와 AI 기반 투자서비스 '맞손'",https://finance.naver.com,연합뉴스TV,10


# contentarea_left > ul > li.newsList.top > dl > dd:nth-child(2)

//*[@id="contentarea_left"]/ul/li[1]/dl/dd[1]

/html/body/div[3]/div[1]/div[2]/div[1]/ul/li[1]/dl/dd[1]

In [309]:
url = 'https://www.chicagomag.com/chicago-magazine/january-2023/our-30-favorite-things-to-eat-right-now/'

In [310]:
resp = requests.get(url)

In [312]:
resp
# 403 접근거부 내일 하는법 알려줌

<Response [403]>